# Vector 기반 GraphRAG (`neo4j-graphrag`)

GraphRAG 를 손으로 다 짜기 전에, **`neo4j-graphrag`** 라이브러리가 제공하는 사전구축 컴포넌트를 쓴다. 이 노트북은 **벡터 검색** 기반: 텍스트를 청킹·임베딩해 Neo4j 에 저장하고, 벡터 유사도로 검색해 답한다.

핵심 3단계:
1. **벡터 인덱스 생성** (`create_vector_index`)
2. **VectorRetriever** 로 유사 청크 검색
3. **GraphRAG** 파이프라인으로 검색+생성

+ **`SimpleKGPipeline`**: PDF 등에서 자동으로 지식그래프(엔티티/관계)를 추출해 적재하는 사전구축 파이프라인.

> Neo4j + `OPENAI_API_KEY` 필요. 실행은 Neo4j 인스턴스가 있어야 한다 (README 참고).

## Neo4j 연결 & LLM
`.env` 에 `NEO4J_URI` / `NEO4J_USERNAME` / `NEO4J_PASSWORD` / `OPENAI_API_KEY` 를 넣는다. (README 의 'Neo4j 준비' 참고)

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
URI = os.environ["NEO4J_URI"]
AUTH = (os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Neo4j 연결 성공")

In [ ]:
from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings

llm = OpenAILLM(model_name="gpt-4o")
embedder = OpenAIEmbeddings()  # text-embedding 계열, 1536차원

## 1. 벡터 인덱스 생성

`Chunk` 노드의 `embedding` 속성에 대한 코사인 유사도 벡터 인덱스를 만든다. (텍스트 청크가 이미 임베딩되어 Neo4j 에 `Chunk` 노드로 저장돼 있다고 가정)

In [ ]:
from neo4j_graphrag.indexes import create_vector_index

INDEX_NAME = "vectorchunk"
create_vector_index(
    driver, INDEX_NAME,
    label="Chunk",
    embedding_property="embedding",
    dimensions=1536,
    similarity_fn="cosine",
)
print("벡터 인덱스 생성:", INDEX_NAME)

## 2. VectorRetriever — 벡터 유사도 검색
질문을 임베딩해 가장 유사한 청크를 찾는다 (일반 RAG 의 벡터 검색과 동일하나, 저장소가 Neo4j).

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever

retriever = VectorRetriever(driver, INDEX_NAME, embedder=embedder)
result = retriever.search(query_text="Who is Marie Curie?", top_k=3)
print(result)

## 3. GraphRAG 파이프라인 — 검색 + 생성
`GraphRAG(retriever, llm)` 가 검색→프롬프트 구성→LLM 생성을 한 번에 처리한다.

In [ ]:
from neo4j_graphrag.generation import GraphRAG

graph_rag = GraphRAG(retriever, llm)
response = graph_rag.search(
    query_text="Who is Marie Curie?",
    retriever_config={"top_k": 3},
    return_context=True,
)
print(response.answer)

## 4. SimpleKGPipeline — PDF → 지식그래프 자동 구축

텍스트에서 **엔티티와 관계를 LLM 으로 추출** 해 그래프로 만드는 사전구축 파이프라인. PDF 를 넣으면 청킹→임베딩→엔티티/관계 추출→Neo4j 적재까지 자동으로 한다.

```python
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

kg_builder = SimpleKGPipeline(
    llm=llm,            # 엔티티/관계 추출용 LLM
    driver=driver,      # 결과를 쓸 Neo4j 드라이버
    embedder=embedder,  # 청크 임베딩용
    from_pdf=True,      # 이미 추출된 텍스트면 False
)
await kg_builder.run_async(file_path="your_document.pdf")
```

적재 후에는 위와 동일하게 `create_vector_index` → `VectorRetriever` → `GraphRAG` 로 검색·생성한다.

## 정리

- `neo4j-graphrag` 는 GraphRAG 를 위한 사전구축 컴포넌트를 제공
- 흐름: 벡터 인덱스 생성 → `VectorRetriever` 검색 → `GraphRAG` 로 생성
- `SimpleKGPipeline` 로 PDF 에서 지식그래프를 자동 구축 가능

다음: 벡터가 아니라 **Cypher 쿼리** 로 그래프를 검색하는 Graph 기반 방식.